# Redrob Hybrid Ranker Sandbox
This sandbox is the hosted, runnable demonstration for the **Redrob Intelligent Candidate Discovery & Ranking Challenge** (Participant ID: `team_204`).

It runs the ranking system end-to-end on a candidate dataset and outputs the ranked shortlist in CSV format according to the evaluation specifications.

## Features
- **Boolean Hard Gate Filtering**: Enforces strict criteria on years of experience, geographic hub/relocation, preferred work mode, role exclusions, and platform activity.
- **Lexical/Semantic Hybrid Retrieval**: Integrates global Okapi BM25, structural token-proximity scoring, and local MiniLM cosine similarity.
- **Behavioral Signal Multiplying**: Applies platform activity curves, notice period penalties, and skill Laplace trust weighting.
- **Anti-Honeypot Trap Matrix**: Automatically detects and drops malicious/fabricated candidate profiles.

### Step 1: Environment Setup
We clone the repository, install the semantic-ranking dependency, and prepare the local MiniLM model used by the ranker.

In [ ]:
import os
# Clone repository if we are running in an external Jupyter/Colab environment
if not os.path.exists("rank_candidates.py"):
    print("Cloning Redrob_Hybrid_Ranker repository...")
    !git clone https://github.com/SaiKarthikMothe/Redrob_Hybrid_Ranker.git
    %cd Redrob_Hybrid_Ranker
else:
    print("Repository already loaded.")

!pip -q install -r requirements.txt
!python prepare_embedding_model.py --out models/all-MiniLM-L6-v2

### Step 2: Running the Ranking Pipeline
You can run the ranker on the preloaded sample dataset (`data/sample_candidates.jsonl`), or choose to upload your own custom JSONL sample using the Google Colab file uploader in the next section.

Let's execute the ranking pipeline on `data/sample_candidates.jsonl`:

In [ ]:
# Run the semantic pipeline on the sample file
!python rank_candidates.py --use-embeddings --embedding-model models/all-MiniLM-L6-v2 --candidates data/sample_candidates.jsonl --out sample_submission.csv

### Step 3: Verifying Output CSV Format
Let's run the official format validator script to check if the generated output meets the challenge requirements.

In [ ]:
# Run the official validator
!python validate_submission.py sample_submission.csv

### Step 4: Displaying the Shortlist
Let's print the ranked shortlist table from the output CSV.

In [ ]:
import pandas as pd
df = pd.read_csv("sample_submission.csv")
pd.set_option('display.max_colwidth', None)
display(df)

### Step 5: (Optional) Upload Your Own Candidate Dataset
If you'd like to test custom candidate profiles, run the cell below to upload your own `.jsonl` or `.json` (JSON array) file. The ranker dynamically detects both formats.

> **Note:** The official validator script (`validate_submission.py`) enforces a strict rule requiring exactly 100 candidates to pass. If your custom uploaded file contains fewer than 100 gate-surviving candidates, the validation step will output a row-count warning. This is expected behavior for custom datasets smaller than the final competition requirement.

In [ ]:
try:
    from google.colab import files
    uploaded = files.upload()
    if uploaded:
        custom_filename = list(uploaded.keys())[0]
        print(f"Uploaded: {custom_filename}")
        # Run the semantic ranker on your custom candidate file
        !python rank_candidates.py --use-embeddings --embedding-model models/all-MiniLM-L6-v2 --candidates {custom_filename} --out custom_submission.csv
        # Validate the custom output
        !python validate_submission.py custom_submission.csv
        # Display results
        df_custom = pd.read_csv("custom_submission.csv")
        display(df_custom)
except ImportError:
    print("google.colab module is not available (this only works inside Google Colab).")